In [1]:
import matplotlib
matplotlib.use('Qt5Agg')  # Important for VSCode interactivity

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import PolygonSelector
from matplotlib.path import Path
from matplotlib.colors import LinearSegmentedColormap
import scanpy as sc

import numpy as np
import os
import plotly.io as pio
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.sparse as sp
from anndata import AnnData

In [2]:
class PolygonAnnotator:
    def __init__(self, image, cmap='hot'):
        self.image = image
        self.coords = None
        self.done = False

        self.fig, self.ax = plt.subplots()
        self.ax.imshow(image, cmap=cmap)
        self.ax.set_title("Click to draw polygon. Double-click to finish.")

        self.selector = PolygonSelector(
            self.ax,
            self.onselect,
            useblit=True,
            props=dict(color='cyan', linewidth=2),
            handle_props=dict(marker='o', markersize=5, mec='cyan', mfc='cyan')
        )

        self.fig.canvas.mpl_connect('close_event', self._on_close)
        plt.show(block=True)  # Blocks until the window is closed

    def _on_close(self, event):
        self.done = True

    def onselect(self, verts):
        self.coords = verts
        print(f"Polygon drawn with {len(verts)} points.")
        plt.close(self.fig)  # Automatically close the window

    def get_mask(self):
        if not self.coords:
            print("No polygon was drawn.")
            return None
        ny, nx = self.image.shape
        y, x = np.mgrid[:ny, :nx]
        points = np.vstack((x.flatten(), y.flatten())).T
        path = Path(self.coords)
        return path.contains_points(points).reshape((ny, nx))

In [3]:
def get_mz_heatmap_image(adata, target_mz, tol=0.1):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values.astype(int)
    y = adata.obs["y"].values.astype(int)

    width = x.max() + 1
    height = y.max() + 1
    image = np.zeros((height, width))
    image[y, x] = intensities

    return image, matched_mz

In [4]:
# Load and visualize the m/z heatmap from AnnData
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad"
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_0.00001_top1000_9w_3p_5points.h5ad


In [5]:
def sort_adata_by_mz(adata, mz_column="mzs"):
    def to_float_with_warning(index, label="var_names"):
        mzs = pd.to_numeric(index, errors="coerce")
        n_nans = mzs.isna().sum()
        if n_nans > 0:
            print(f"⚠️ Warning: {n_nans} entries in {label} could not be converted to float and were set to NaN.")
        return mzs
    adata.var[mz_column] = to_float_with_warning(adata.var_names)
    adata = adata[:, adata.var[mz_column].sort_values().index]
    return adata


In [6]:
adata = sort_adata_by_mz(adata, mz_column="mzs")
adata.var

,mzs
136.0150,136.0150
136.0608,136.0608
137.0251,137.0251
138.0287,138.0287
139.0359,139.0359
...,...
1520.1588,1520.1588
1521.1633,1521.1633
1522.1682,1522.1682
1542.1475,1542.1475


In [7]:
# ---- Run ----

target_mz = 788.5
image, matched_mz = get_mz_heatmap_image(adata, target_mz)
print(f"Target m/z: {target_mz} → Matched m/z: {matched_mz}")

custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

#custom_cmap = "binary"
# Use the annotator with your image and colormap
annotator = PolygonAnnotator(image, cmap=custom_cmap)
mask = annotator.get_mask()

if mask is not None:
    masked_image = np.where(mask, image, 0)
    plt.imshow(masked_image, cmap=custom_cmap)
    plt.title(f"Masked m/z = {matched_mz:.4f}")
    plt.show()
else:
    print("No mask created.")

Target m/z: 788.5 → Matched m/z: 788.5027
Polygon drawn with 28 points.


In [8]:
# Get spatial coordinates
x = adata.obs["x"].astype(int).values
y = adata.obs["y"].astype(int).values

# Mask value at each spot
adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)
adata.obs["tissue"] = adata.obs["tissue"].astype(int)  # 1 = inside, 0 = outside
print(adata.obs["tissue"].value_counts())

tissue
0    9467
1    7138
Name: count, dtype: int64


/var/folders/0n/w0dydx896p7b5chql0_cv6hr0000gn/T/ipykernel_48875/1596079847.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["tissue"] = mask[y, x]  # Boolean mask (True = inside polygon)


In [9]:
adata1 = adata
adata.obs["tissue"]

0        0
1        0
2        0
3        0
4        0
        ..
16600    0
16601    0
16602    0
16603    0
16604    0
Name: tissue, Length: 16605, dtype: int64

In [10]:
adata = adata1
adata

AnnData object with n_obs × n_vars = 16605 × 1000
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'tissue'
    var: 'mzs'

In [11]:
def plot_mz_across_sample_custom_tic(adata, target_mz, tol=0.1):
    """
    Plots the TIC-normalized intensity of a specific m/z across all pixels using spatial coordinates.

    Parameters:
    - adata: AnnData object with MSI data
    - target_mz: float, the m/z value of interest
    - tol: float, tolerance for matching m/z (default ±0.1)
    """
    # Convert var_names to float (m/z axis)
    mz_axis = adata.var_names.astype(float).values

    # Find index of the m/z closest to the target within tolerance
    mz_diff = np.abs(mz_axis - target_mz)
    if np.min(mz_diff) > tol:
        raise ValueError(f"No m/z found within ±{tol} of {target_mz}")
    mz_index = np.argmin(mz_diff)
    matched_mz = mz_axis[mz_index]

    # Extract intensities for that m/z from all pixels
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]

    # Normalize by TIC
    if "TIC" not in adata.obs.columns:
        raise KeyError("TIC not found in adata.obs. Compute TIC before using this function.")
    tic = adata.obs["TIC"].values
    norm_intensities = intensities / tic
    norm_intensities[np.isnan(norm_intensities)] = 0  # Avoid NaNs from division by 0

    # Extract spatial coordinates
    if "x" not in adata.obs.columns or "y" not in adata.obs.columns:
        raise KeyError("Spatial coordinates 'x' and 'y' not found in adata.obs")

    x = adata.obs["x"].values
    y = adata.obs["y"].values

    # Create DataFrame for plotting
    df = pd.DataFrame({"x": x, "y": y, "intensity": norm_intensities})

    # Plot as heatmap using plotly express
    fig = px.scatter(
        df,
        x="x",
        y="y",
        color="intensity",
        color_continuous_scale=[
            [0.0, "#000000"],  # black
            [0.10, "#000080"],  # navy
            [0.15, "#0000FF"],  # blue
            [0.30, "#8000FF"],  # purple-ish
            [0.45, "#FF0000"],  # red
            [0.60, "#FF8000"],  # orange
            [0.75, "#FFFF00"],  # yellow
            [1.0, "#FFFFFF"]   # white
        ],
        title=f"TIC-Normalized Intensity Map of m/z = {matched_mz:.4f}",
        labels={"intensity": "Normalized Intensity"},
        height=500
    )

    fig.update_layout(yaxis=dict(scaleanchor="x", autorange="reversed"))
    return fig

In [12]:
"""
Moran's I and Spatial Entropy Analysis
This code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   
Moran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. 
Moran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.
moran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.
Spatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   
spatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.
The analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.
The DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.
"""

"\nMoran's I and Spatial Entropy Analysis\nThis code computes Moran's I and spatial entropy for each m/z feature in the provided AnnData object.   \nMoran's I is a measure of spatial autocorrelation, while spatial entropy quantifies the distribution of intensities. \nMoran's I values close to 1 indicate clustering, while values close to -1 indicate dispersion.\nmoran's I is computed using the `libpysal` library, which requires spatial coordinates and intensities.\nSpatial entropy values range from 0 (uniform distribution) to log(n) (maximal entropy), where n is the number of pixels.   \nspatial entropy is computed using the `scipy.stats.entropy` function, which requires the input to be a probability distribution.\nThe analysis is performed separately for tissue and background regions, and results are stored in a DataFrame.\nThe DataFrame contains the m/z values, spatial entropy, and Moran's I for both regions.\n"

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
from tqdm import tqdm

# Define global offsets for common adducts and isotopes
ADDUCTS_AND_ISOTOPES = {
    "has_H_adduct": 1.0073,
    "has_Na_adduct": 22.9898,
    "has_K_adduct": 38.9637,
    "has_Mplus1_isotope": 1.00335,
    "has_Mplus2_isotope": 2.0067,
}

def find_related_mz_features(mz_values, current_mz, tolerance=0.01):
    related_features = {}
    for label, offset in ADDUCTS_AND_ISOTOPES.items():
        target_mz = current_mz + offset
        match = np.any(np.isclose(mz_values, target_mz, atol=tolerance))
        related_features[label] = int(match)
    return related_features

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        # Uncomment to debug
        print(f"Moran's I calculation failed: {e}")
        return np.nan

def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception as e:
        # Uncomment to debug
        print(f"Geary's C calculation failed: {e}")
        return np.nan

def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                # No spots for this region
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            # Filter out zero intensities for stats like CV and skew only if nonzero values exist
            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        related_features = find_related_mz_features(mz_values, mz_values[mz_index])
        row.update(related_features)

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics

In [14]:
'''
# Run analysis
df_metrics = analyze_metrics(adata, k_neighbors=8)

# Drop rows with NaNs if needed (optional)
df_metrics_clean = df_metrics.dropna()

# Normalize & scale
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df_metrics_clean.columns if col != "mz"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])

# UMAP dimensionality reduction
import umap
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_scaled)

# t-SNE dimensionality reduction
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# Clustering
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(X_scaled)

# Add to DataFrame
df_metrics_clean["cluster"] = clusters
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]
df_metrics_clean["tSNE1"] = X_tsne[:, 0]
df_metrics_clean["tSNE2"] = X_tsne[:, 1]

print(df_metrics_clean.head())'''

'\n# Run analysis\ndf_metrics = analyze_metrics(adata, k_neighbors=8)\n\n# Drop rows with NaNs if needed (optional)\ndf_metrics_clean = df_metrics.dropna()\n\n# Normalize & scale\nfrom sklearn.preprocessing import StandardScaler\nfeature_cols = [col for col in df_metrics_clean.columns if col != "mz"]\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(df_metrics_clean[feature_cols])\n\n# UMAP dimensionality reduction\nimport umap\nreducer = umap.UMAP(random_state=42)\nX_umap = reducer.fit_transform(X_scaled)\n\n# t-SNE dimensionality reduction\nfrom sklearn.manifold import TSNE\ntsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)\nX_tsne = tsne.fit_transform(X_scaled)\n\n# Clustering\nimport hdbscan\nclusterer = hdbscan.HDBSCAN(min_cluster_size=5)\nclusters = clusterer.fit_predict(X_scaled)\n\n# Add to DataFrame\ndf_metrics_clean["cluster"] = clusters\ndf_metrics_clean["UMAP1"] = X_umap[:, 0]\ndf_metrics_clean["UMAP2"] = X_umap[:, 1]\ndf_metrics_clean["tSNE

In [15]:
'''from sklearn.cluster import MiniBatchKMeans

# Choose number of clusters
n_clusters = 100  # or any number between 100-200

# Run clustering
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
df_metrics_clean["cluster"] = kmeans.fit_predict(X_scaled)'''

'from sklearn.cluster import MiniBatchKMeans\n\n# Choose number of clusters\nn_clusters = 100  # or any number between 100-200\n\n# Run clustering\nkmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)\ndf_metrics_clean["cluster"] = kmeans.fit_predict(X_scaled)'

In [16]:
'''for cluster_id, group in df_metrics_clean.groupby("cluster"):
    print(f"\nCluster {cluster_id} ({len(group)} m/z features):")
    print(group["mz"].values)'''

'for cluster_id, group in df_metrics_clean.groupby("cluster"):\n    print(f"\nCluster {cluster_id} ({len(group)} m/z features):")\n    print(group["mz"].values)'

In [17]:
#clustered_mzs = df_metrics_clean.groupby("cluster")["mz"].apply(list).to_dict()


In [18]:
#df_metrics_clean[["mz", "cluster"]].sort_values("cluster").to_csv("clustered_mzs.csv", index=False)


In [19]:
#cluster_counts = df_metrics_clean["cluster"].value_counts().sort_index()
#print(cluster_counts)

In [20]:
"""import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts/cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")"""

'import os\nimport plotly.io as pio\n\n# Ensure the image folder exists\nos.makedirs(f"images_from_clusters_adducts", exist_ok=True)\n\n# Merge cluster info with AnnData\'s m/z values\nmz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster"]))\n\n# Compute the global spectrum\nglobal_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()\n\n# Get indices of m/z values sorted by intensity (descending)\nsorted_indices = np.argsort(global_spectrum)[::-1]\n\n# Select indices of interest (e.g., all, top N, etc.)\nselected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired\n\n# Get corresponding m/z values from adata.var\nselected_mz = adata.var["mzs"].values[selected_indices].astype(float)\n\n# Generate and save images\nfor mz in selected_mz:\n    if mz not in mz_cluster_map:\n        print(f"Skipping m/z {mz:.4f} — not in clustered data.")\n        continue\n\n    cluster_id = mz_clust

In [21]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
from tqdm import tqdm

# Define global offsets for common adducts and isotopes
ADDUCTS_AND_ISOTOPES = {
    "has_H_adduct": 1.0073,
    "has_Na_adduct": 22.9898,
    "has_K_adduct": 38.9637,
    "has_Mplus1_isotope": 1.00335,
    "has_Mplus2_isotope": 2.0067,
}

'''def find_related_mz_features(mz_values, current_mz, tolerance=0.01):
    related_features = {}
    for label, offset in ADDUCTS_AND_ISOTOPES.items():
        target_mz = current_mz + offset
        match = np.any(np.isclose(mz_values, target_mz, atol=tolerance))
        related_features[label] = int(match)
    return related_features'''

'''def build_mz_relation_matrix(mz_values, tolerance=0.01):
    n = len(mz_values)
    relation_matrix = np.zeros((n, n), dtype=int)

    for i, mz_i in enumerate(mz_values):
        for j, mz_j in enumerate(mz_values):
            if i == j:
                continue
            for offset in ADDUCTS_AND_ISOTOPES.values():
                if np.isclose(mz_j, mz_i + offset, atol=tolerance):
                    relation_matrix[i, j] = 1
                    break  # Avoid redundant checks
    return relation_matrix'''

def build_mz_relation_matrix(mz_values, tolerance=0.01):
    n = len(mz_values)
    relation_matrix = np.zeros((n, n), dtype=int)

    for i, mz_i in enumerate(mz_values):
        for j, mz_j in enumerate(mz_values):
            if i == j:
                continue
            for offset in ADDUCTS_AND_ISOTOPES.values():
                if np.isclose(mz_j, mz_i + offset, atol=tolerance):
                    relation_matrix[i, j] = 1
                    break  # Avoid redundant checks
    return relation_matrix

def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        print(f"Moran's I calculation failed: {e}")
        return np.nan

def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception as e:
        print(f"Geary's C calculation failed: {e}")
        return np.nan

def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        #related_features = find_related_mz_features(mz_values, mz_values[mz_index])
        #row.update(related_features)

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics

In [22]:
# ---------- Analysis ----------
df_metrics = analyze_metrics(adata, k_neighbors=8)

# Drop rows with NaNs
df_metrics_clean = df_metrics.dropna()

# Build relationship matrix
mz_values_clean = df_metrics_clean["mz"].values
relation_matrix = build_mz_relation_matrix(mz_values_clean)

# Normalize spatial/statistical features
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df_metrics_clean.columns if col not in ["mz", "cluster", "UMAP1", "UMAP2", "tSNE1", "tSNE2"]]
X_stat = df_metrics_clean[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_stat)

# Combine with relation matrix
X_combined = np.hstack([X_scaled, relation_matrix])

# Dimensionality reduction
import umap
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_combined)

from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_combined)

# Clustering (HDBSCAN)
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(X_combined)

# Add to DataFrame
df_metrics_clean["cluster"] = clusters
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]
df_metrics_clean["tSNE1"] = X_tsne[:, 0]
df_metrics_clean["tSNE2"] = X_tsne[:, 1]

print(df_metrics_clean.head())


Processing m/z features: 100%|██████████| 1000/1000 [19:09<00:00,  1.15s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


         mz  moran_I_tissue  geary_C_tissue  entropy_tissue  cv_tissue  \
0  136.0150        0.324070        0.641172        8.431539   0.881417   
1  136.0608        0.237098        0.751356        8.707171   0.611926   
2  137.0251        0.397854        0.538474        8.652862   0.831213   
3  138.0287        0.376210        0.568720        8.626295   0.823157   
4  139.0359        0.271404        0.701928        8.553374   0.768983   

   skew_tissue  region_size  moran_I_background  geary_C_background  \
0     2.875010         6180            0.482480            0.511093   
1     1.814193         7131            0.501375            0.494571   
2     6.687794         7138            0.719608            0.277547   
3     4.906731         7109            0.698279            0.298135   
4     2.759772         6619            0.646160            0.350097   

   entropy_background  cv_background  skew_background  cluster      UMAP1  \
0            9.059755       0.425373         0.3291

In [23]:
# Optional: MiniBatch KMeans clustering
from sklearn.cluster import MiniBatchKMeans
n_clusters = 100
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
df_metrics_clean["cluster_kmeans"] = kmeans.fit_predict(X_combined)

In [24]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts_relation_matrix", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts_relation_matrix/cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_273.0397_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_137.0251_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_274.0435_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_369.3523_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_331.0297_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_257.0478_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_195.0905_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_637.3020_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_313.0337_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_275.0563_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_245.0458_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_-1_mz_333.0266_.png
Saved: images_from_clusters_adducts_rela

In [25]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, variation, skew
from libpysal.weights import KNN
from esda.moran import Moran
from esda.geary import Geary
from tqdm import tqdm



def compute_spatial_entropy(intensities):
    intensities = np.array(intensities, dtype=float)
    intensities = intensities[intensities > 0]
    if len(intensities) == 0:
        return np.nan
    p = intensities / intensities.sum()
    return entropy(p)

def compute_morans_I(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Moran(intensities, w).I
    except Exception as e:
        print(f"Moran's I calculation failed: {e}")
        return np.nan

def compute_gearys_C(x_coords, y_coords, intensities, k=8):
    coords = list(zip(x_coords, y_coords))
    try:
        w = KNN(coords, k=k)
        return Geary(intensities, w).C
    except Exception as e:
        print(f"Geary's C calculation failed: {e}")
        return np.nan

def analyze_metrics(adata, k_neighbors=8):
    num_mz = adata.shape[1]
    mz_values = adata.var["mzs"].values

    rows = []
    for mz_index in tqdm(range(num_mz), desc="Processing m/z features"):
        row = {"mz": mz_values[mz_index]}

        for flag, region in zip([True, False], ["tissue", "background"]):
            subset = adata[adata.obs["tissue"] == flag]
            if subset.shape[0] == 0:
                row[f"moran_I_{region}"] = np.nan
                row[f"geary_C_{region}"] = np.nan
                row[f"entropy_{region}"] = np.nan
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan
                row[f"region_size"] = 0 if region == "tissue" else np.nan
                continue

            intensities = subset.X[:, mz_index].toarray().flatten() if hasattr(subset.X, "toarray") else subset.X[:, mz_index].flatten()
            x = subset.obs["x"].values
            y = subset.obs["y"].values

            nonzero_intensities = intensities[intensities > 0]

            row[f"moran_I_{region}"] = compute_morans_I(x, y, intensities, k=k_neighbors)
            row[f"geary_C_{region}"] = compute_gearys_C(x, y, intensities, k=k_neighbors)
            row[f"entropy_{region}"] = compute_spatial_entropy(intensities)

            if len(nonzero_intensities) > 0:
                row[f"cv_{region}"] = variation(nonzero_intensities)
                row[f"skew_{region}"] = skew(nonzero_intensities)
            else:
                row[f"cv_{region}"] = np.nan
                row[f"skew_{region}"] = np.nan

            if region == "tissue":
                row[f"region_size"] = np.sum(intensities > 0)

        #related_features = find_related_mz_features(mz_values, mz_values[mz_index])
        #row.update(related_features)

        rows.append(row)

    df_metrics = pd.DataFrame(rows)
    return df_metrics

In [26]:
# ---------- Analysis ----------
df_metrics = analyze_metrics(adata, k_neighbors=8)

# Drop rows with NaNs
df_metrics_clean = df_metrics.dropna()

# Normalize spatial/statistical features
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df_metrics_clean.columns if col not in ["mz", "cluster", "UMAP1", "UMAP2", "tSNE1", "tSNE2"]]
X_stat = df_metrics_clean[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_stat)

# Dimensionality reduction
import umap
reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X_scaled)

from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# Add to DataFrame
df_metrics_clean["UMAP1"] = X_umap[:, 0]
df_metrics_clean["UMAP2"] = X_umap[:, 1]
df_metrics_clean["tSNE1"] = X_tsne[:, 0]
df_metrics_clean["tSNE2"] = X_tsne[:, 1]

print(df_metrics_clean.head())

Processing m/z features: 100%|██████████| 1000/1000 [19:47<00:00,  1.19s/it]
/opt/anaconda3/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



         mz  moran_I_tissue  geary_C_tissue  entropy_tissue  cv_tissue  \
0  136.0150        0.324070        0.641172        8.431539   0.881417   
1  136.0608        0.237098        0.751356        8.707171   0.611926   
2  137.0251        0.397854        0.538474        8.652862   0.831213   
3  138.0287        0.376210        0.568720        8.626295   0.823157   
4  139.0359        0.271404        0.701928        8.553374   0.768983   

   skew_tissue  region_size  moran_I_background  geary_C_background  \
0     2.875010         6180            0.482480            0.511093   
1     1.814193         7131            0.501375            0.494571   
2     6.687794         7138            0.719608            0.277547   
3     4.906731         7109            0.698279            0.298135   
4     2.759772         6619            0.646160            0.350097   

   entropy_background  cv_background  skew_background     UMAP1      UMAP2  \
0            9.059755       0.425373         0.329

In [27]:
#MiniBatch KMeans clustering
from sklearn.cluster import MiniBatchKMeans
n_clusters = 100
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
df_metrics_clean["cluster_kmeans"] = kmeans.fit_predict(X_scaled)

In [28]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts_relation_matrix", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster_kmeans"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts_relation_matrix/cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters_adducts_relation_matrix/cluster_86_mz_273.0397_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_86_mz_137.0251_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_86_mz_274.0435_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_10_mz_369.3523_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_22_mz_331.0297_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_0_mz_257.0478_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_62_mz_195.0905_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_62_mz_637.3020_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_57_mz_313.0337_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_57_mz_275.0563_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_57_mz_245.0458_.png
Saved: images_from_clusters_adducts_relation_matrix/cluster_22_mz_333.0266_.png
Saved: images_from_clusters_adducts_relat

In [29]:
# Define global offsets for common adducts and isotopes
ADDUCTS_AND_ISOTOPES = {
    "has_H_adduct": 1.0073,
    "has_Na_adduct": 22.9898,
    "has_K_adduct": 38.9637,
    "has_Mplus1_isotope": 1.00335,
    "has_Mplus2_isotope": 2.0067,
}
def build_mz_relation_matrix(mz_values, tolerance=0.01):
    n = len(mz_values)
    relation_matrix = np.zeros((n, n), dtype=int)

    for i, mz_i in enumerate(mz_values):
        for j, mz_j in enumerate(mz_values):
            if i == j:
                relation_matrix[i, j] = 1  # Each m/z is related to itself
                continue
            for offset in ADDUCTS_AND_ISOTOPES.values():
                if np.isclose(mz_j, mz_i + offset, atol=tolerance) or np.isclose(mz_i, mz_j + offset, atol=tolerance):
                    relation_matrix[i, j] = 1
                    relation_matrix[j, i] = 1  # Ensure symmetry
                    break
    return relation_matrix

In [30]:
mz_values_clean = df_metrics_clean["mz"].values
relation_matrix = build_mz_relation_matrix(mz_values_clean, tolerance=0.01)


In [31]:
from sklearn.cluster import SpectralClustering
from collections import defaultdict

# Initialize dictionary to store sub-cluster labels
subclusters = np.full_like(df_metrics_clean["cluster_kmeans"], fill_value=-1, dtype=int)

# Map from mz to index for matrix referencing
mz_to_index = {mz: i for i, mz in enumerate(df_metrics_clean["mz"].values)}

# Iterate through each KMeans cluster
for cluster_id in np.unique(df_metrics_clean["cluster_kmeans"]):
    # Get the m/z values in this cluster
    cluster_mask = df_metrics_clean["cluster_kmeans"] == cluster_id
    mz_values_cluster = df_metrics_clean.loc[cluster_mask, "mz"].values

    if len(mz_values_cluster) < 3:
        continue  # Skip tiny clusters

    # Get indices of these m/zs in the full relation matrix
    indices = [mz_to_index[mz] for mz in mz_values_cluster]
    submatrix = relation_matrix[np.ix_(indices, indices)]

    # Use Spectral Clustering on the submatrix
    n_subclusters = min(5, len(indices))  # You can tune this
    spectral = SpectralClustering(
        n_clusters=n_subclusters,
        affinity='precomputed',
        assign_labels='kmeans',
        random_state=42
    )
    labels = spectral.fit_predict(submatrix)

    # Assign subcluster labels (optionally encode with main cluster id)
    full_labels = [f"{cluster_id}_{label}" for label in labels]
    subclusters[cluster_mask] = full_labels

# Add to DataFrame
df_metrics_clean["subcluster"] = subclusters

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning:

Graph is not fully connected, spectral embedding may not work as expected.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:372: RuntimeWarning:

k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning

In [32]:
df_metrics_clean

,mz,moran_I_tissue,geary_C_tissue,entropy_tissue,cv_tissue,skew_tissue,region_size,moran_I_background,geary_C_background,entropy_background,cv_background,skew_background,UMAP1,UMAP2,tSNE1,tSNE2,cluster_kmeans,subcluster
0,136.0150,0.324070,0.641172,8.431539,0.881417,2.875010,6180,0.482480,0.511093,9.059755,0.425373,0.329159,4.404281,6.019695,-0.439980,-19.475126,29,294
1,136.0608,0.237098,0.751356,8.707171,0.611926,1.814193,7131,0.501375,0.494571,8.672948,1.130451,5.384610,-0.038512,10.060963,11.304199,15.373377,89,893
2,137.0251,0.397854,0.538474,8.652862,0.831213,6.687794,7138,0.719608,0.277547,8.953559,0.629648,0.607490,0.960604,4.464797,22.255169,-19.615423,86,862
3,138.0287,0.376210,0.568720,8.626295,0.823157,4.906731,7109,0.698279,0.298135,8.993100,0.563046,0.526112,1.062554,4.493981,20.804996,-17.942438,86,862
4,139.0359,0.271404,0.701928,8.553374,0.768983,2.759772,6619,0.646160,0.350097,9.017759,0.520332,0.556648,2.566808,5.662715,12.584943,-14.435235,99,994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1520.1588,0.277004,0.717376,8.673543,0.656378,1.738991,7103,0.601680,0.375352,7.934591,1.518004,4.070000,-1.449665,10.754025,11.237035,27.564888,6,60
996,1521.1633,0.283741,0.710725,8.659486,0.670377,1.461790,7084,0.602655,0.378950,7.952759,1.452838,3.828118,-1.454702,10.714418,11.495955,27.854225,6,60
997,1522.1682,0.287201,0.708156,8.663615,0.673610,1.877519,7103,0.584486,0.391175,7.950907,1.466803,3.878655,-1.445360,10.740431,11.657845,27.371542,6,60
998,1542.1475,0.234540,0.765401,8.678258,0.639450,1.610881,7092,0.521071,0.446395,8.086660,1.196134,3.124230,-1.000650,10.679148,10.582418,24.639679,8,84


In [33]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts_relation_matrix_sub", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["subcluster"]))
mz_cluster_map_kmeans = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster_kmeans"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    cluster_id_kmeans = mz_cluster_map_kmeans[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts_relation_matrix_sub/{cluster_id_kmeans}_cluster_{cluster_id}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters_adducts_relation_matrix_sub/86_cluster_861_mz_273.0397_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/86_cluster_862_mz_137.0251_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/86_cluster_864_mz_274.0435_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/10_cluster_101_mz_369.3523_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/22_cluster_220_mz_331.0297_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/0_cluster_1_mz_257.0478_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/62_cluster_622_mz_195.0905_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/62_cluster_620_mz_637.3020_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/57_cluster_574_mz_313.0337_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/57_cluster_573_mz_275.0563_.png
Saved: images_from_clusters_adducts_relation_matrix_sub/57_cluster_571_mz_245.0458_.png
Saved: images_from_clusters_adducts

In [34]:
import os
import plotly.io as pio

# Ensure the image folder exists
os.makedirs(f"images_from_clusters_adducts_relation_matrix_sub_v2", exist_ok=True)

# Merge cluster info with AnnData's m/z values
mz_cluster_map = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["subcluster"]))
mz_cluster_map_kmeans = dict(zip(df_metrics_clean["mz"].astype(float), df_metrics_clean["cluster_kmeans"]))

# Compute the global spectrum
global_spectrum = adata.X.sum(axis=0).A1 if hasattr(adata.X, "A1") else np.asarray(adata.X.sum(axis=0)).flatten()

# Get indices of m/z values sorted by intensity (descending)
sorted_indices = np.argsort(global_spectrum)[::-1]

# Select indices of interest (e.g., all, top N, etc.)
selected_indices = sorted_indices[0:]  # You can limit with [10:20] if desired

# Get corresponding m/z values from adata.var
selected_mz = adata.var["mzs"].values[selected_indices].astype(float)

# Generate and save images
for mz in selected_mz:
    if mz not in mz_cluster_map:
        print(f"Skipping m/z {mz:.4f} — not in clustered data.")
        continue

    cluster_id = mz_cluster_map[mz]
    cluster_id_kmeans = mz_cluster_map_kmeans[mz]
    fig = plot_mz_across_sample_custom_tic(adata, mz, tol=0.001)  # Make sure this function returns a Plotly fig

    filename = f"images_from_clusters_adducts_relation_matrix_sub_v2/{cluster_id}_cluster_{cluster_id_kmeans}_mz_{mz:.4f}_.png"
    pio.write_image(fig, filename)
    print(f"Saved: {filename}")

Saved: images_from_clusters_adducts_relation_matrix_sub_v2/861_cluster_86_mz_273.0397_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/862_cluster_86_mz_137.0251_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/864_cluster_86_mz_274.0435_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/101_cluster_10_mz_369.3523_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/220_cluster_22_mz_331.0297_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/1_cluster_0_mz_257.0478_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/622_cluster_62_mz_195.0905_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/620_cluster_62_mz_637.3020_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/574_cluster_57_mz_313.0337_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/573_cluster_57_mz_275.0563_.png
Saved: images_from_clusters_adducts_relation_matrix_sub_v2/571_cluster_57_mz_245.0458_.png
Sa